# Build-up phase: Effective Playing Space and formation stretching

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

In the build-up phase within the midfield zone, the "Effective Playing Space" of both teams is examined using the Convex Hull algorithm, alongside the degree of play stretching, assessed through an analysis of formation centroids.

This notebook runs the shared pipeline (`pitchvision`: detection, tracking, pitch calibration, team classification - see `00_pipeline_demo.ipynb` for a step-by-step walkthrough with sanity checks, and `01_goal_scoring_opportunity.ipynb` for the jersey-colour clustering audit tools) over **every clip in the `BuildingAction` folder**, not just one - each clip gets its own saved CSVs, and a per-clip failure (bad calibration, too few jersey samples) is skipped with a warning rather than stopping the whole run:

1. **Effective Playing Space**: each team's own convex hull (the smallest polygon containing all of that team's outfield players) as well as both teams combined (the classical Frencken et al. definition) - how much of the pitch is actively being used.
2. **Degree of stretching**: each team's own formation centroid/stretch (reusing `pitchvision.compactness`), plus the distance *between* the two teams' centroids - how far the build-up pulls the two blocks apart, split into the pitch's length-axis and width-axis components.

Batch results (one row per clip) are saved to `buildup_all_clips_summary.csv`; full per-frame CSVs are saved per clip as before. A separate section at the end lets you inspect one clip's plots closely without re-running the pipeline.

## 1. Setup

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/field-position-detection-dqda1g"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

# Put the package on sys.path directly, rather than relying on `pip install -e .`
# to register it - editable installs use a .pth/import-finder file that Python's
# site module only processes at interpreter startup, so one run mid-session (in
# an already-running Colab kernel) doesn't reliably become importable.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Drive and list clips

In [ ]:
from pitchvision import DriveConfig, list_videos, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")
buildup_videos = list_videos(drive_cfg.building_action_path)
print(f"Found {len(buildup_videos)} videos in BuildingAction/:")
for path in buildup_videos:
    print(" ", os.path.basename(path))

## 3. Shared model setup (downloaded once)

Model weights don't need re-downloading per clip - only pitch calibration and team classification
are actually clip-specific. The detector/tracker themselves are still rebuilt fresh *inside* the
per-clip function below (see `process_buildup_clip`) rather than reused across clips: Ultralytics'
`model.track(..., persist=True)` keeps tracker state (track IDs, Kalman filters) alive across calls
on the *same* model instance, which would leak track identity from one clip into the next if the
same `PlayerTracker` were reused - the same pattern `03_set_pieces.ipynb` already uses for exactly
this reason.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from pitchvision import (
    PitchKeypointDetector,
    PlayerBallDetector,
    PlayerTracker,
    SPORTS_DETECTION_CLASSES,
    TeamClassifier,
    TrackingPipeline,
    VideoFrames,
    collect_jersey_samples,
    compute_centroid_separation,
    compute_combined_convex_hull,
    compute_team_compactness,
    compute_team_convex_hull,
    download_pitch_keypoint_weights,
    download_player_detection_weights,
    draw_pitch,
    plot_convex_hull,
    plot_jersey_color_samples,
    save_convex_hull_frames,
)

pitch_weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)
keypoint_detector = PitchKeypointDetector(weights=pitch_weights_path, confidence=0.5)

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
print("Shared weights ready.")

## 4. Process every clip

`silhouette_score` (see `04_validation.ipynb`) gives a zero-labeling sanity check on each clip's
team-colour split - low/negative means that clip's jersey-colour clustering probably isn't finding
real team structure, worth auditing with `plot_jersey_color_samples` in the inspect-one-clip
section below.

In [ ]:
def process_buildup_clip(video_path, output_dir, max_frames=250):
    frames = VideoFrames(video_path)
    first_frame = frames.read_frame(0)
    calibrator = keypoint_detector.calibrate(first_frame)

    detector = PlayerBallDetector(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    jersey_samples = collect_jersey_samples(video_path, detector, class_names=("player",), stride=15)
    jersey_colors = np.array([s.color for s in jersey_samples])
    team_classifier = TeamClassifier(n_clusters=2).fit(jersey_colors)

    cluster_labels = team_classifier.predict_from_colors(jersey_colors)
    silhouette = float("nan")
    if len(set(cluster_labels)) > 1:
        scaled = StandardScaler().fit_transform(jersey_colors)
        silhouette = silhouette_score(scaled, cluster_labels)

    tracker = PlayerTracker(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    pipeline = TrackingPipeline(
        tracker=tracker,
        calibrator=calibrator,
        team_classifier=team_classifier,
        team_eligible_class_names=("player",),
    )
    tracks_df = pipeline.run(video_path, max_frames=max_frames)

    hull_team0 = compute_team_convex_hull(tracks_df, team_id=0)
    hull_team1 = compute_team_convex_hull(tracks_df, team_id=1)
    hull_combined = compute_combined_convex_hull(tracks_df)
    compactness0 = compute_team_compactness(tracks_df, team_id=0)
    compactness1 = compute_team_compactness(tracks_df, team_id=1)
    separation = compute_centroid_separation(tracks_df, team_a_id=0, team_b_id=1)

    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    hull_team0.to_csv(os.path.join(output_dir, f"{clip_name}_hull_team0.csv"), index=False)
    hull_team1.to_csv(os.path.join(output_dir, f"{clip_name}_hull_team1.csv"), index=False)
    hull_combined.to_csv(os.path.join(output_dir, f"{clip_name}_hull_combined.csv"), index=False)
    compactness0.to_csv(os.path.join(output_dir, f"{clip_name}_compactness_team0.csv"), index=False)
    compactness1.to_csv(os.path.join(output_dir, f"{clip_name}_compactness_team1.csv"), index=False)
    separation.to_csv(os.path.join(output_dir, f"{clip_name}_centroid_separation.csv"), index=False)

    summary = {
        "clip": clip_name,
        "n_frames": tracks_df["frame"].nunique(),
        "jersey_cluster_silhouette": silhouette,
        "mean_hull_team0_m2": hull_team0["area_m2"].mean(),
        "mean_hull_team1_m2": hull_team1["area_m2"].mean(),
        "mean_hull_combined_m2": hull_combined["area_m2"].mean(),
        "mean_stretch_team0_m": compactness0["stretch_index_m"].mean(),
        "mean_stretch_team1_m": compactness1["stretch_index_m"].mean(),
        "mean_centroid_distance_m": separation["centroid_distance_m"].mean(),
    }
    return {
        "summary": summary,
        "tracks_df": tracks_df,
        "hull_team0": hull_team0,
        "hull_team1": hull_team1,
        "hull_combined": hull_combined,
        "compactness0": compactness0,
        "compactness1": compactness1,
        "separation": separation,
        "jersey_samples": jersey_samples,
        "team_classifier": team_classifier,
        "team_colors": {0: "yellow", 1: "red"},
    }

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)

results_by_clip = {}
summaries = []
failed = []
for video_path in buildup_videos:
    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f"Processing {clip_name}...")
    try:
        result = process_buildup_clip(video_path, output_dir)
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        failed.append({"clip": clip_name, "error": str(e)})
        continue
    results_by_clip[clip_name] = result
    summaries.append(result["summary"])
    print(f"  done - {result['summary']['n_frames']} frames tracked")

summary_df = pd.DataFrame(summaries)
summary_path = os.path.join(output_dir, "buildup_all_clips_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nProcessed {len(summaries)}/{len(buildup_videos)} clips successfully. Summary saved to {summary_path}")
if failed:
    print("Skipped clips:", failed)
summary_df

## 5. Inspect one clip closely

Batch processing above intentionally skips plotting (one clip's worth of figures is useful, thirty
clips' worth is just noise) and the heavier per-frame PNG export. Pick a clip from the batch results
above to look at in detail - no need to re-run the pipeline, this reuses the cached result.

In [ ]:
INSPECT_CLIP = next(iter(results_by_clip))  # change to inspect a different clip - see results_by_clip.keys()
print("Inspecting:", INSPECT_CLIP)

result = results_by_clip[INSPECT_CLIP]
tracks_df = result["tracks_df"]
hull_team0, hull_team1, hull_combined = result["hull_team0"], result["hull_team1"], result["hull_combined"]
compactness0, compactness1, separation = result["compactness0"], result["compactness1"], result["separation"]
jersey_samples, team_classifier = result["jersey_samples"], result["team_classifier"]
TEAM_COLORS = result["team_colors"]
print(f"silhouette score: {result['summary']['jersey_cluster_silhouette']:.3f}")

swatches = team_classifier.cluster_swatches
fig, axes = plt.subplots(1, len(swatches), figsize=(4 * len(swatches), 2))
for team_id, (ax, rgb) in enumerate(zip(axes, swatches)):
    ax.imshow([[rgb]])
    ax.set_title(f"team_id = {team_id}")
    ax.axis("off")
plt.show()

**Audit the fit before trusting it.** See `01_goal_scoring_opportunity.ipynb` for what to look for here - two visually separated blobs of roughly similar size in the scatter/crop grid below is trustworthy; one tight blob plus a handful of scattered outliers is not.

In [ ]:
cluster_labels = team_classifier.predict_from_colors(np.array([s.color for s in jersey_samples]))
plot_jersey_color_samples(jersey_samples, cluster_labels, cluster_colors={0: "yellow", 1: "red"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hull_team0["frame"], hull_team0["area_m2"], label="Team 0", color=TEAM_COLORS[0])
ax.plot(hull_team1["frame"], hull_team1["area_m2"], label="Team 1", color=TEAM_COLORS[1])
ax.plot(hull_combined["frame"], hull_combined["area_m2"], label="Combined (Effective Playing Space)", color="black", linestyle="--")
ax.set_xlabel("Frame")
ax.set_ylabel("Convex hull area (m\u00b2)")
ax.set_title(f"Effective Playing Space over the phase - {INSPECT_CLIP}")
ax.legend()
plt.show()

print("Team 0 hull area (m2):", hull_team0["area_m2"].describe()[["mean", "min", "max"]].to_dict())
print("Team 1 hull area (m2):", hull_team1["area_m2"].describe()[["mean", "min", "max"]].to_dict())
print("Combined EPS (m2):", hull_combined["area_m2"].describe()[["mean", "min", "max"]].to_dict())

In [ ]:
# A representative frame: the one with the most on-pitch players tracked.
on_pitch = tracks_df[(tracks_df["class_name"] == "player") & tracks_df["team_id"].notna()]
sample_frame = on_pitch.groupby("frame").size().idxmax()
frame_rows = on_pitch[on_pitch["frame"] == sample_frame]

ax = draw_pitch()
for team_id, group in frame_rows.groupby("team_id"):
    positions = group[["pitch_x", "pitch_y"]].to_numpy()
    color = TEAM_COLORS.get(team_id, "gray")
    plot_convex_hull(ax, positions, color=color)
    ax.scatter(positions[:, 0], positions[:, 1], color=color, edgecolors="black", s=60, zorder=3)
plt.title(f"Effective Playing Space at frame {sample_frame} - {INSPECT_CLIP}")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(compactness0["frame"], compactness0["stretch_index_m"], label="Team 0", color=TEAM_COLORS[0])
ax1.plot(compactness1["frame"], compactness1["stretch_index_m"], label="Team 1", color=TEAM_COLORS[1])
ax1.set_xlabel("Frame")
ax1.set_ylabel("Stretch index (m)")
ax1.set_title("Each team's own formation stretch")
ax1.legend()

ax2.plot(separation["frame"], separation["length_axis_separation_m"], label="Length-axis (goal-to-goal)")
ax2.plot(separation["frame"], separation["width_axis_separation_m"], label="Width-axis (touchline-to-touchline)")
ax2.plot(separation["frame"], separation["centroid_distance_m"], label="Overall distance", linestyle="--", color="black")
ax2.set_xlabel("Frame")
ax2.set_ylabel("Metres")
ax2.set_title("Separation between the two teams' centroids")
ax2.legend()
plt.tight_layout()
plt.show()

### Optional: batch-export Effective Playing Space diagrams for this clip

Off by default - `save_convex_hull_frames` renders one PNG per frame, which adds up fast across
many clips, so this only ever runs for the single inspected clip above, not the whole batch.

In [ ]:
EXPORT_HULL_FRAMES = False

if EXPORT_HULL_FRAMES:
    hull_dir = os.path.join(output_dir, f"{INSPECT_CLIP}_convex_hull")
    saved_paths = save_convex_hull_frames(tracks_df, hull_dir, stride=10, team_colors=TEAM_COLORS)
    print(f"Saved {len(saved_paths)} Effective Playing Space diagrams to {hull_dir}")
else:
    print("Skipping PNG export (EXPORT_HULL_FRAMES is False).")

## Notes and limitations

- **Every clip in `BuildingAction/` is processed**, not just one - a clip that fails (bad
  calibration, too few jersey samples for a 2-cluster fit) is skipped with its error printed to
  `failed`, rather than stopping the whole batch. Check `len(summaries)` against
  `len(buildup_videos)` and the printed skip list before treating `buildup_all_clips_summary.csv`
  as complete.
- Convex hull area only counts the *outermost* players - two teams with the same hull area can have very different internal density (e.g. one bunched near the hull's edges with an empty midfield, another evenly spread). Cross-reference with `stretch_index_m`/`mean_pairwise_distance_m` (`pitchvision.compactness`) for a fuller picture of a team's actual shape, not just its outer boundary.
- Goalkeepers are excluded from both the convex hull and compactness metrics by default, since a goalkeeper anchored deep in their own box would distort a midfield/build-up shape analysis that isn't about them.
- The combined Effective Playing Space can shrink even while one team's own hull grows, if the two teams' players are moving into overlapping space rather than spreading apart - the per-team and combined figures answer different questions and are both worth reporting.
- `jersey_cluster_silhouette` in the summary is a *zero-labeling* proxy for team-split quality (see `04_validation.ipynb`) - low/negative values are worth auditing with `plot_jersey_color_samples` in section 5, not necessarily discarding outright.